In [13]:
# -- Downlaod locally
# https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v1/tree/main

# gif lfs install
# git config --global http.sslBackend schannel
# git clone https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v1

In [14]:
print("LangChain, Chroma, and Hugging Face form a powerful, entirely open-source tech stack commonly used to build Retrieval-Augmented Generation (RAG) systems")

LangChain, Chroma, and Hugging Face form a powerful, entirely open-source tech stack commonly used to build Retrieval-Augmented Generation (RAG) systems


In [15]:
# !pip install sentence-transformers
# !pip install langchain langchain-chroma langchain-huggingface langchain-community faiss-cpu sentence-transformers
# !pip install pypdf

In [16]:
# https://wikidocs.net/234094

In [17]:
import os
os.environ['CURL_CA_BUNDLE'] = ''

In [18]:
from IPython.display import HTML

In [19]:
# # !pip install pysqlite3-binary 
# langchain-chroma error regarding pysqlite3
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [20]:
# SentenceTransformerEmbeddings: SBERT 모델을 활용한 문장 임베딩 생성
# from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from typing import List
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
# FAISS: 효율적인 유사성 검색을 위한 벡터 데이터베이스 라이브러리
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_openai import OpenAIEmbeddings
# from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import uuid
import json
import shutil
import re

### Vector Database
* __Embedding Model (Vector Conversion)__: 텍스트를 고차원 숫자 벡터로 변환합니다. 변환된 벡터들은 의미가 비슷할수록 벡터 공간상에 가깝게 배치되며, 주로 인코더(Encoder) 중심의 구조를 띱니다 (An Embedding Model is designed to translate data (like text) into numerical lists (vectors) that represent semantic meaning, making it ideal for similarity searches)
* __Embedding__ : Hugging Face Embedding,  Google Generative Embedding, OpenAI Embedding
* __Chroma__ : most commonly refers to the open-source vector database used in AI and machine learning. It allows developers to easily store, embed, and query large collections of documents, images, and data for Retrieval-Augmented Generation (RAG) and LLM applications. (Vector DB : FAISS, ANNoy, Elasticsearch and etc..)

In [21]:
def process_pages(pages: List[Document]) -> List[Document]:
    def transform_trim_string(to_replace): 
        """remove unnecessary characters"""
        if isinstance(to_replace, (str)):
            to_replace = to_replace.strip()
            to_replace = re.sub(r'(?<!\.)(\n|\r\n)', ' ', to_replace)
            to_replace = re.sub(r'\t|\\t', ' ', to_replace)
            to_replace = re.sub(r' +', ' ', to_replace)

        return to_replace
         
    return [Document(page_content=transform_trim_string(page.page_content), metadata=page.metadata) for page in pages]

In [22]:
# 1. Load your local open-source data
# loader = TextLoader("your_data.txt")
# documents = loader.load()

In [23]:
# 1. 문서 리스트 준비
sentences = [
    "우리나라는 2022년에 코로나가 유행했다.",
    "우리나라 2024년 GDP 전망은 3.0%이다.",
    "우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.",
    "LangChain is a robust framework built to orchestrate LLM applications.",
    "FAISS (Facebook AI Similarity Search) is a highly efficient library for clustering and similarity search of dense vectors.",
    "Chroma is a highly efficient, open-source vector database tailored for AI applications.",
    "Hugging Face is a massive platform hosting thousands of open-source machine learning models."
]

In [24]:
def convert_documents_to_dataframe(docs):
    # Convert LangChain Documents to a list of dictionaries
    data = [
        {
            "page_content": doc.page_content,
            "id" : doc.id,
            **doc.metadata  # Unpacks metadata keys directly into columns [OR]
            # "mdeta_data" : doc.metadata
        } 
        for doc in docs
    ]

    # print(data)

    # Create the Pandas DataFrame
    df = pd.DataFrame(data)
    # print(df.head(30))
    display(df)

In [25]:
# Wrap the strings into LangChain Document objects
# documents = [Document(page_content=text, metadata=dict(page=i+1, id=str(uuid.uuid4()))) for i, text in enumerate(sentences)]
documents = [Document(page_content=text, metadata={"page" : i+1, "id" : str(uuid.uuid4())}) for i, text in enumerate(sentences)]

In [26]:
documents.append(
    Document(
        page_content="pgvector is an open-source extension that adds vector search directly into an existing PostgreSQL" , 
        metadata=dict(page=10, id=str(uuid.uuid4()))
    )
)

In [27]:
documents

[Document(metadata={'page': 1, 'id': '1a14ebc1-73a5-4a05-89cf-7c6b3b6e099f'}, page_content='우리나라는 2022년에 코로나가 유행했다.'),
 Document(metadata={'page': 2, 'id': '2d958daa-dada-468a-a848-4357cc3a159e'}, page_content='우리나라 2024년 GDP 전망은 3.0%이다.'),
 Document(metadata={'page': 3, 'id': '229de753-d7c6-459a-8179-86ec397d7678'}, page_content='우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.'),
 Document(metadata={'page': 4, 'id': '61d53f0f-e15f-46cc-857e-863b177dc959'}, page_content='LangChain is a robust framework built to orchestrate LLM applications.'),
 Document(metadata={'page': 5, 'id': '10c5981b-c988-4916-90b7-6b9f200a7b3f'}, page_content='FAISS (Facebook AI Similarity Search) is a highly efficient library for clustering and similarity search of dense vectors.'),
 Document(metadata={'page': 6, 'id': '7738560a-88d5-4f30-b398-b3bc1f4347c3'}, page_content='Chroma is a highly efficient, open-source vector database tailored for AI applications.'),
 Document(metadata={'page': 7, 'id': '6eb30bf9-fb69-4aae-8a6

In [28]:
# Printout Dataframe format after traonsforming from Document
convert_documents_to_dataframe(documents)

,page_content,id,page
0,우리나라는 2022년에 코로나가 유행했다.,1a14ebc1-73a5-4a05-89cf-7c6b3b6e099f,1
1,우리나라 2024년 GDP 전망은 3.0%이다.,2d958daa-dada-468a-a848-4357cc3a159e,2
2,우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.,229de753-d7c6-459a-8179-86ec397d7678,3
3,LangChain is a robust framework built to orche...,61d53f0f-e15f-46cc-857e-863b177dc959,4
4,FAISS (Facebook AI Similarity Search) is a hig...,10c5981b-c988-4916-90b7-6b9f200a7b3f,5
5,"Chroma is a highly efficient, open-source vect...",7738560a-88d5-4f30-b398-b3bc1f4347c3,6
6,Hugging Face is a massive platform hosting tho...,6eb30bf9-fb69-4aae-8a6c-0a4bbe2c5451,7
7,pgvector is an open-source extension that adds...,54161d2d-8e09-4307-8863-6b13f919ed13,10


In [29]:
# 1. 임베딩 모델 준비
# embeddings = OpenAIEmbeddings()

# 2. SBERT 모델 초기화
# 사용 모델: distiluse-base-multilingual-cased-v1 (다국어 지원 모델)
# embedding = SentenceTransformerEmbeddings(model_name="distiluse-base-multilingual-cased-v1")

# 2. 임베딩 모델 로드 (한국어 지원 모델)
# 허깅페이스의 sbert-korean-base 모델을 사용합니다.
# embedding = SentenceTransformer('jhgan/ko-sroberta-multitask')

In [30]:

# 1. Download the model from Hugging Face
# model_name = "sentence-transformers/all-MiniLM-L6-v2"
# model = SentenceTransformer(model_name)

# 2. Save it to a local directory
# local_path = "./my_local_embedding_model"
# model.save(local_path)

# print(f"Model successfully saved offline to: {local_path}")

In [31]:
print("LangChain: The orchestrator framework. It coordinates document loading, splitting, vector database queries, and sending the final prompt to an LLM")
print("Hugging Face: The embedding and LLM provider. It supplies open-source text models")
print("Chroma (ChromaDB): The AI-native vector database. It stores those mathematical vectors locally or in the cloud and performs fast similarity searches")

LangChain: The orchestrator framework. It coordinates document loading, splitting, vector database queries, and sending the final prompt to an LLM
Hugging Face: The embedding and LLM provider. It supplies open-source text models
Chroma (ChromaDB): The AI-native vector database. It stores those mathematical vectors locally or in the cloud and performs fast similarity searches


In [32]:
print("Below is a complete, minimal script showcasing how to split text, embed it locally via Hugging Face, save it to a local Chroma database, and query it.")

Below is a complete, minimal script showcasing how to split text, embed it locally via Hugging Face, save it to a local Chroma database, and query it.


In [33]:
# Add file for testing
# from text_loader import loader_text
from langchain_community.document_loaders import PyPDFLoader

In [34]:
pdf_filepath = './files/asiabrief_3-26.pdf'
loader = PyPDFLoader(pdf_filepath)
data = loader.load()

In [35]:
data = process_pages(data)
# print(data)
data = [Document(page_content=text.page_content, metadata=dict(page=text.metadata.get("page"), id=str(uuid.uuid4()))) for i, text in enumerate(data)]

In [36]:
# 2. Split documents into small, manageable chunks
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = text_splitter.split_documents(data)

In [37]:
documents.extend(docs)

In [38]:
convert_documents_to_dataframe(documents)

,page_content,id,page
0,우리나라는 2022년에 코로나가 유행했다.,1a14ebc1-73a5-4a05-89cf-7c6b3b6e099f,1
1,우리나라 2024년 GDP 전망은 3.0%이다.,2d958daa-dada-468a-a848-4357cc3a159e,2
2,우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.,229de753-d7c6-459a-8179-86ec397d7678,3
3,LangChain is a robust framework built to orche...,61d53f0f-e15f-46cc-857e-863b177dc959,4
4,FAISS (Facebook AI Similarity Search) is a hig...,10c5981b-c988-4916-90b7-6b9f200a7b3f,5
5,"Chroma is a highly efficient, open-source vect...",7738560a-88d5-4f30-b398-b3bc1f4347c3,6
6,Hugging Face is a massive platform hosting tho...,6eb30bf9-fb69-4aae-8a6c-0a4bbe2c5451,7
7,pgvector is an open-source extension that adds...,54161d2d-8e09-4307-8863-6b13f919ed13,10
8,Seoul National University Asia Center 1 2023년 ...,54c587f7-327a-4aa5-b7bc-b4bf9fae79b2,0
9,소득과 자원에 따라 출산율은 0명까지 감소할 가능성을 배제할 수 없다. 베커의 신가...,54c587f7-327a-4aa5-b7bc-b4bf9fae79b2,0


In [39]:
# 2. Split documents into small, manageable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

In [40]:
# docs

In [41]:
print(f"Vector search is a modern retrieval technique that finds information based on semantic meaning rather than exact keyword matches. Items with similar meanings or characteristics are clustered close together (e.g., 'cat' and 'feline'). When you run a query, the system converts your search into a vector and mathematically calculates which stored items are closest.")

Vector search is a modern retrieval technique that finds information based on semantic meaning rather than exact keyword matches. Items with similar meanings or characteristics are clustered close together (e.g., 'cat' and 'feline'). When you run a query, the system converts your search into a vector and mathematically calculates which stored items are closest.


In [42]:
# --
# Hugging Face가 인터넷을 검색하지 않도록 오프라인 환경 변수 설정
# os.environ["HF_HUB_OFFLINE"] = "1"
# os.environ["TRANSFORMERS_OFFLINE"] = "1"

# 로컬에 저장된 모델 경로 불러오기
model_path = "../huggingfase_model/distiluse-base-multilingual-cased-v1/"

# model = SentenceTransformer(model_path)
# # 3. Calculate embeddings by calling model.encode()
# embeddings = model.encode(sentences, convert_to_numpy=True)
# print(f"Embeddings shape: {embeddings.shape}")

embedding_model = HuggingFaceEmbeddings(
    model_name=model_path
)
print(f"Embeddings: {embedding_model}")
# --

Embeddings: model_name='../huggingfase_model/distiluse-base-multilingual-cased-v1/' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [43]:
# 2. 기존 DB 폴더 삭제 (초기화)
# 'ignore_errors=True'로 설정하면 폴더가 이미 없어도 에러가 발생하지 않습니다.

# IS_INDEXING = True
IS_INDEXING = False

# Chroma DB delete and save 
DB_DIR="./chroma_db"
if IS_INDEXING:
    if os.path.exists(DB_DIR):
        shutil.rmtree(DB_DIR)

In [44]:
# document_1 = Document(page_content="foo", metadata={"baz": "bar"})
# document_2 = Document(page_content="thud", metadata={"bar": "baz"})
# document_3 = Document(page_content="i will be deleted :(")
# ids = ["1", "2", "3"]
# db.add_documents(documents=documents, ids=ids)

### Chroma DB Insert/Update/Search
* If you want to perform a strict keyword search without using vector embeddings or calculating similarity metrics, use the __<i>collection.get() method alongside the $contains or $not_contains</i>__
* This complete example demonstrates __<i>how to split text into documents, convert them into vector embeddings, and build the Chroma database</i>__

In [45]:
# --
# Onnly Text
# 문서 정의
# texts = ["랭체인은 LLM 앱 개발 프레임워크입니다.", "Chroma는 오픈소스 벡터 데이터베이스입니다."]
# db = Chroma.from_texts(
#     texts=texts, 
#     embedding=embeddings, 
#     collection_name="my_first_collection",
#     persist_directory="./chroma_db"
# )
# --

# --
# - doc_list의 각 문장을 벡터화하여 FAISS 데이터베이스에 저장
# - 각 문서에 메타데이터를 부여하여 추적 가능하도록 설정
# vector_store = FAISS.from_texts(
#     sentences, embeddings, metadatas=[{"source": 1}] * len(doc_list)
# )

# Step 3: Create the FAISS index and add the documents
print("Generating embeddings and building Vector index...")
# Step 3: Create and populate the Chroma Vector Database
# vector_store = FAISS.from_documents(docs, embedding_model)

''' https://reference.langchain.com/python/langchain-chroma/vectorstores/Chroma '''
if IS_INDEXING:
    ''' Indexing '''
    db = Chroma.from_documents(documents=documents, embedding=embedding_model, persist_directory=DB_DIR)
    print("Index successfully created!")
else:
    db = Chroma(embedding_function=embedding_model, persist_directory=DB_DIR)
    print("Loaded the Chroma DB..")
# --

Generating embeddings and building Vector index...
Loaded the Chroma DB..


In [46]:
# -- Get all docs from Chroma DB 
# db.get()

In [47]:
# page_content, metadata, id 지정
# db.add_documents(
#     [
#         Document(
#             page_content="안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼께요",
#             metadata={"source": "mydata.txt"},
#             id="1",
#         )
#     ]
# )

In [48]:
# 저장된 데이터 확인
# db.get()

In [49]:
# Update the document in Chroma using its ID
print(f"Update Document Start..")
db.update_document(
    document_id="1",
    document=Document(
        page_content="안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼께요~UPDATE",
        # metadata={"source": "manual_v2.pdf", "status": "updated"}
        metadata={'source': 'mydata.txt'}
    )
)

# Delete items using their document IDs
# db.delete(ids=["doc_id_1", "doc_id_2"])

Update Document Start..


In [50]:
print(f"Update Document Finished.. -> page_content = {db.get(ids=['1']).get('documents')}")

Update Document Finished.. -> page_content = ['안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼께요~UPDATE']


In [51]:
# Retrieves LangChain Document objects matching the given IDs
db.get(ids=["1"])

{'ids': ['1'],
 'embeddings': None,
 'documents': ['안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼께요~UPDATE'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'mydata.txt'}]}

In [52]:
# If you want to perform a strict keyword search without using vector embeddings or calculating similarity metrics, use the collection.get() method alongside the $contains or $not_contains operators
# Query records containing a specific keyword
results = db.get(
    where_document={"$contains": " 도큐먼트"}
)

results["documents"]

['안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼께요~UPDATE']

In [53]:
# 5. 검색 쿼리 입력
# query = "2022년 우리나라 GDP대비 R&D 규모는?"
query = "What is the tool used for storing vectors?"
# query = "한국의 출산율은 얼마쯤 되나요?"

In [54]:
query_result = embedding_model.embed_query(query)
query_result[:3]
# query_result

[0.08492680639028549, -0.05665656924247742, 0.009867217391729355]

In [55]:
docs = db.similarity_search(query, k=3)
# print(docs)

In [56]:
# Printout Dataframe format after traonsforming from Document
convert_documents_to_dataframe(docs)

,page_content,id,page
0,pgvector is an open-source extension that adds...,e3cbf52b-36d6-45d0-b330-d7f41bedbd87,10
1,FAISS (Facebook AI Similarity Search) is a hig...,f9dafcd2-35a2-4800-b67e-a943b7b42e16,5
2,"Chroma is a highly efficient, open-source vect...",b8bbcfc7-7bf1-4db5-8224-b6f7dae46a0b,6


In [57]:
print(docs[0].page_content)

pgvector is an open-source extension that adds vector search directly into an existing PostgreSQL


In [58]:
# 4. 검색기(FaissRetriever) 설정
# - k=1: 가장 유사한 문장 1개만 검색
vector_store = db.as_retriever(search_kwargs={"k": 3})
# vector_store = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3})

In [59]:
# 6. 유사 문서 검색 수행
docs = vector_store.invoke(query)

In [60]:
# 7. 검색 결과 출력
# print(docs)
# 결과: "우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다." 문장이 가장 유사한 결과로 반환

In [61]:
# Printout Dataframe format after traonsforming from Document
convert_documents_to_dataframe(docs)

,page_content,id,page
0,pgvector is an open-source extension that adds...,e3cbf52b-36d6-45d0-b330-d7f41bedbd87,10
1,FAISS (Facebook AI Similarity Search) is a hig...,f9dafcd2-35a2-4800-b67e-a943b7b42e16,5
2,"Chroma is a highly efficient, open-source vect...",b8bbcfc7-7bf1-4db5-8224-b6f7dae46a0b,6


In [62]:
# Step 6: Print out the best match
print("Top Match Content:")
print(docs[0].page_content)

Top Match Content:
pgvector is an open-source extension that adds vector search directly into an existing PostgreSQL


In [63]:
HTML('<pre style="background-color: #A52A2A; color: white;">Job is being performed correctly</pre>')